# 03 · Modelo Tabular (baseline)

**Objetivo:** Establecer un modelo baseline usando únicamente los metadatos clínicos (edad, sexo, localización) sin información visual.

**Datos de entrada:** `../data/raw/HAM10000_metadata.csv`

**Resultado esperado:** Modelo guardado en `../models/tabular_model.h5` con ~69% de accuracy en test.

**Arquitectura:** Red densa (Sequential) — 3 capas Dense con Dropout.

## Carga de datos y preprocesado

El preprocesado (split 70/15/15, imputación, estandarización) está centralizado en `utils.py`.
Las decisiones de diseño están documentadas en `02_data_preparation.ipynb`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ── Carga y preprocesado centralizados en utils.py ────────────────────────────
from utils import get_tabular_splits

X_train, X_val, X_test, y_train, y_val, y_test, label_encoder = get_tabular_splits()
num_classes = y_train.shape[1]

# ── Arquitectura ──────────────────────────────────────────────────────────────
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ── Entrenamiento ─────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

# ── Evaluación ────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")
model.save('../models/tabular_model.h5')

plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Accuracy — Modelo Tabular')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend(); plt.show()

plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Loss — Modelo Tabular')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()

## Evaluación detallada por clase

El accuracy global no es suficiente en diagnóstico médico. Aquí vemos cuánto acierta el modelo en **cada tipo de lesión** por separado, y lo comparamos con el **baseline ZeroR** (lo que conseguiría un modelo que predice siempre la clase más frecuente, sin aprender nada).

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predicciones sobre el conjunto de test
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Baseline ZeroR: accuracy de predecir siempre la clase más frecuente
from collections import Counter
most_common = Counter(y_true).most_common(1)[0][1]
baseline = most_common / len(y_true)
print(f'Baseline ZeroR (predecir siempre la clase más frecuente): {baseline:.2%}')
print(f'Accuracy del modelo: {(y_pred == y_true).mean():.2%}')
print(f'Mejora sobre baseline: {(y_pred == y_true).mean() - baseline:+.2%}')

# Desglose por clase
print('
Resultados por clase diagnóstica:')
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

# Conclusiones del modelo tabular

Vemos que este modelo es muy limitado, ya que su precisión es aproximadamente de 0.69%. Esto es de esperar, ya que utiliza sólo metadatos de la lesión (la localización de la lesión, edad del paciente y su sexo), no la imagen real. Sin embargo, podría ser un buen punto de entrada para plantear un modelo Early o Late fusion donde utilicemos estos datos con un peso menor para ayudar a nuestras redes a entender mejor los datos de las imágenes. Además, nos sirve para comparar con los modelos basados sólo en imágenes y observar su efectividad
Además, como podemos ver en los gráficos, a partir de la epoch 10 conseguimos una efectividad bastante estable, por lo que probablemente limitar las epoch a 10 habría sido un buen movimiento (no lo hacemos porque al ser sólo metadatos, nuestras cpus y gpus ) hacen este trabajo con relativa facilidad y no nos supone mucho más tiempo)